In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 - Generate Synthetic Healthcare Data
# MAGIC
# MAGIC This notebook generates synthetic healthcare data for the Healthcare Risk Data Lakehouse project.
# MAGIC
# MAGIC The generated datasets include:
# MAGIC - Members
# MAGIC - Providers
# MAGIC - Labs
# MAGIC - Medications
# MAGIC - Claims
# MAGIC
# MAGIC The data is saved to DBFS so it can be used by the Bronze ingestion notebook.

In [0]:
# Databricks notebook source
import random
import uuid
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [0]:
PROJECT_NAME = "healthcare-risk-data-lakehouse"

# Unity Catalog Volumes paths (serverless compatible)
VOLUME_BASE_PATH = "/Volumes/workspace/default/healthcare_data"
RAW_DATA_PATH = f"{VOLUME_BASE_PATH}/raw"

# Local driver path for pandas writes (Volumes accessible via /Volumes)
LOCAL_RAW_DATA_PATH = f"{VOLUME_BASE_PATH}/raw"

# Legacy DBFS paths kept for reference (read-only on serverless)
DBFS_BASE_PATH = f"dbfs:/FileStore/{PROJECT_NAME}"

print(f"Volume base path: {VOLUME_BASE_PATH}")
print(f"Raw data path: {RAW_DATA_PATH}")
print(f"Local raw path: {LOCAL_RAW_DATA_PATH}")

In [0]:
# Create the raw data directory inside the Unity Catalog Volume
dbutils.fs.mkdirs(RAW_DATA_PATH)

# Confirm the project folder was created
display(dbutils.fs.ls(VOLUME_BASE_PATH))

In [0]:
random.seed(42)
np.random.seed(42)

In [0]:
def random_date(start_date, end_date):
    """
    Generate a random date between start_date and end_date.
    """
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    delta = end - start
    random_days = random.randint(0, delta.days)
    return start + timedelta(days=random_days)


def calculate_age(date_of_birth, reference_date="2024-08-31"):
    """
    Calculate age based on a reference date.
    """
    dob = pd.to_datetime(date_of_birth)
    ref = pd.to_datetime(reference_date)
    return ref.year - dob.year - ((ref.month, ref.day) < (dob.month, dob.day))


def introduce_nulls(df, column_name, null_rate=0.02):
    """
    Randomly introduce null values into a column.
    """
    df = df.copy()
    mask = np.random.rand(len(df)) < null_rate
    df.loc[mask, column_name] = None
    return df


def save_to_dbfs(df, file_name):
    """
    Save a pandas DataFrame to DBFS as a CSV file.
    """
    output_path = f"{LOCAL_RAW_DATA_PATH}/{file_name}"
    df.to_csv(output_path, index=False)
    print(f"Saved {file_name} to {output_path}")

In [0]:
num_members = 10000

first_names = [
    "James", "Mary", "John", "Patricia", "Robert", "Jennifer", "Michael",
    "Linda", "William", "Elizabeth", "David", "Barbara", "Richard", "Susan"
]

last_names = [
    "Smith", "Johnson", "Williams", "Brown", "Jones", "Garcia", "Miller",
    "Davis", "Rodriguez", "Martinez", "Hernandez", "Lopez", "Gonzalez"
]

genders = ["Male", "Female", "M", "F", "male", "female", "Unknown"]
races = ["Black", "White", "Hispanic", "Asian", "Other", "Unknown"]
zip_codes = ["70801", "70802", "70805", "70806", "70808", "70810", "70501", "70503"]

members = []

for i in range(1, num_members + 1):
    dob = random_date("1940-01-01", "2005-12-31").date()
    enrollment_start = random_date("2020-01-01", "2024-01-01").date()
    
    if random.random() < 0.85:
        enrollment_end = None
    else:
        enrollment_end = random_date("2024-01-01", "2024-12-31").date()
    
    members.append({
        "member_id": f"M{i:06d}",
        "first_name": random.choice(first_names),
        "last_name": random.choice(last_names),
        "date_of_birth": dob,
        "gender": random.choice(genders),
        "race": random.choice(races),
        "zip_code": random.choice(zip_codes),
        "enrollment_start_date": enrollment_start,
        "enrollment_end_date": enrollment_end
    })

members_df = pd.DataFrame(members)

# Add duplicate records for data quality testing
duplicate_members = members_df.sample(100, random_state=42)
members_df = pd.concat([members_df, duplicate_members], ignore_index=True)

# Introduce null values
members_df = introduce_nulls(members_df, "gender", 0.02)
members_df = introduce_nulls(members_df, "zip_code", 0.01)

display(spark.createDataFrame(members_df.head(10)))
print(f"Members record count: {len(members_df)}")

In [0]:
num_providers = 1000

specialties = [
    "Primary Care", "Endocrinology", "Cardiology", "Nephrology",
    "Internal Medicine", "Family Medicine", "Emergency Medicine"
]

cities = ["Baton Rouge", "Lafayette", "New Orleans", "Shreveport", "Lake Charles"]
states = ["LA", "la", "Louisiana"]

providers = []

for i in range(1, num_providers + 1):
    providers.append({
        "provider_id": f"P{i:05d}",
        "provider_name": f"{random.choice(last_names)} Medical Group",
        "specialty": random.choice(specialties),
        "city": random.choice(cities),
        "state": random.choice(states)
    })

providers_df = pd.DataFrame(providers)

# Add duplicates
duplicate_providers = providers_df.sample(25, random_state=42)
providers_df = pd.concat([providers_df, duplicate_providers], ignore_index=True)

display(spark.createDataFrame(providers_df.head(10)))
print(f"Providers record count: {len(providers_df)}")

In [0]:
num_labs = 50000

lab_types = ["HbA1c", "Cholesterol", "Glucose", "Creatinine"]
member_ids = members_df["member_id"].dropna().unique().tolist()

labs = []

for i in range(1, num_labs + 1):
    member_id = random.choice(member_ids)
    lab_type = random.choices(
        lab_types,
        weights=[0.65, 0.15, 0.15, 0.05],
        k=1
    )[0]
    
    if lab_type == "HbA1c":
        # Most HbA1c values are controlled, but some are uncontrolled
        lab_value = round(np.random.normal(7.4, 1.8), 1)
        
        # Force some values above 9 for uncontrolled diabetes
        if random.random() < 0.20:
            lab_value = round(np.random.uniform(9.1, 13.5), 1)
    elif lab_type == "Cholesterol":
        lab_value = round(np.random.normal(190, 35), 1)
    elif lab_type == "Glucose":
        lab_value = round(np.random.normal(130, 40), 1)
    else:
        lab_value = round(np.random.normal(1.1, 0.3), 1)
    
    labs.append({
        "lab_id": f"L{i:07d}",
        "member_id": member_id,
        "lab_date": random_date("2024-01-01", "2024-08-31").date(),
        "lab_type": lab_type,
        "lab_value": lab_value
    })

labs_df = pd.DataFrame(labs)

# Add invalid HbA1c values for Silver layer cleaning
invalid_labs = pd.DataFrame([
    {
        "lab_id": "L_INVALID_001",
        "member_id": random.choice(member_ids),
        "lab_date": "2024-04-15",
        "lab_type": "HbA1c",
        "lab_value": 2.1
    },
    {
        "lab_id": "L_INVALID_002",
        "member_id": random.choice(member_ids),
        "lab_date": "2024-05-20",
        "lab_type": "HbA1c",
        "lab_value": 25.0
    }
])

labs_df = pd.concat([labs_df, invalid_labs], ignore_index=True)

# Add duplicates
duplicate_labs = labs_df.sample(250, random_state=42)
labs_df = pd.concat([labs_df, duplicate_labs], ignore_index=True)

# Introduce nulls
labs_df = introduce_nulls(labs_df, "lab_value", 0.01)

display(spark.createDataFrame(labs_df.head(10)))
print(f"Labs record count: {len(labs_df)}")

In [0]:
num_medications = 30000

medication_names = [
    "Metformin",
    "Insulin",
    "Glipizide",
    "Jardiance",
    "Ozempic",
    "Trulicity",
    "Lisinopril",
    "Atorvastatin"
]

medications = []

for i in range(1, num_medications + 1):
    medications.append({
        "medication_id": f"RX{i:07d}",
        "member_id": random.choice(member_ids),
        "medication_name": random.choice(medication_names),
        "fill_date": random_date("2024-01-01", "2024-08-31").date(),
        "days_supply": random.choice([30, 60, 90]),
        "adherence_flag": random.choices([1, 0], weights=[0.78, 0.22], k=1)[0]
    })

medications_df = pd.DataFrame(medications)

# Add invalid adherence values
invalid_medications = pd.DataFrame([
    {
        "medication_id": "RX_INVALID_001",
        "member_id": random.choice(member_ids),
        "medication_name": "Metformin",
        "fill_date": "2024-02-10",
        "days_supply": 30,
        "adherence_flag": 3
    },
    {
        "medication_id": "RX_INVALID_002",
        "member_id": random.choice(member_ids),
        "medication_name": "Insulin",
        "fill_date": "2024-03-10",
        "days_supply": 30,
        "adherence_flag": -1
    }
])

medications_df = pd.concat([medications_df, invalid_medications], ignore_index=True)

# Add duplicates
duplicate_medications = medications_df.sample(150, random_state=42)
medications_df = pd.concat([medications_df, duplicate_medications], ignore_index=True)

# Introduce nulls
medications_df = introduce_nulls(medications_df, "member_id", 0.005)

display(spark.createDataFrame(medications_df.head(10)))
print(f"Medications record count: {len(medications_df)}")

In [0]:
num_claims = 75000

provider_ids = providers_df["provider_id"].dropna().unique().tolist()

diagnosis_codes = {
    "diabetes": ["E11.9", "E11.65", "E10.9"],
    "hypertension": ["I10", "I11.9"],
    "ckd": ["N18.1", "N18.2", "N18.3", "N18.4"],
    "general": ["Z00.00", "R53.83", "J06.9", "M54.5"]
}

place_of_service_options = [
    "Office",
    "Hospital",
    "Emergency Room",
    "Telehealth",
    "Urgent Care"
]

claims = []

all_diagnosis_codes = (
    diagnosis_codes["diabetes"]
    + diagnosis_codes["hypertension"]
    + diagnosis_codes["ckd"]
    + diagnosis_codes["general"]
)

for i in range(1, num_claims + 1):
    diagnosis_code = random.choices(
        all_diagnosis_codes,
        weights=[0.18, 0.15, 0.12, 0.10, 0.08, 0.07, 0.05, 0.03, 0.03, 0.06, 0.05, 0.04, 0.04],
        k=1
    )[0]
    
    place_of_service = random.choice(place_of_service_options)
    
    if place_of_service == "Hospital":
        claim_amount = round(np.random.uniform(1000, 15000), 2)
    elif place_of_service == "Emergency Room":
        claim_amount = round(np.random.uniform(500, 5000), 2)
    else:
        claim_amount = round(np.random.uniform(75, 750), 2)
    
    claims.append({
        "claim_id": f"C{i:08d}",
        "member_id": random.choice(member_ids),
        "provider_id": random.choice(provider_ids),
        "claim_date": random_date("2024-01-01", "2024-08-31").date(),
        "diagnosis_code": diagnosis_code,
        "claim_amount": claim_amount,
        "place_of_service": place_of_service
    })

claims_df = pd.DataFrame(claims)

# Add negative claim amounts for Silver layer cleaning
invalid_claims = pd.DataFrame([
    {
        "claim_id": "C_INVALID_001",
        "member_id": random.choice(member_ids),
        "provider_id": random.choice(provider_ids),
        "claim_date": "2024-04-01",
        "diagnosis_code": "E11.65",
        "claim_amount": -250.00,
        "place_of_service": "Office"
    },
    {
        "claim_id": "C_INVALID_002",
        "member_id": random.choice(member_ids),
        "provider_id": None,
        "claim_date": "2024-05-01",
        "diagnosis_code": "I10",
        "claim_amount": 175.00,
        "place_of_service": "Office"
    }
])

claims_df = pd.concat([claims_df, invalid_claims], ignore_index=True)

# Add duplicates
duplicate_claims = claims_df.sample(500, random_state=42)
claims_df = pd.concat([claims_df, duplicate_claims], ignore_index=True)

# Introduce null provider IDs
claims_df = introduce_nulls(claims_df, "provider_id", 0.01)

display(spark.createDataFrame(claims_df.head(10)))
print(f"Claims record count: {len(claims_df)}")

In [0]:
def save_to_volume(df, filename):    df.to_csv(filename, index=False)
save_to_volume(members_df, "members.csv")

save_to_volume(providers_df, "providers.csv")
save_to_volume(labs_df, "labs.csv")
save_to_volume(medications_df, "medications.csv")
save_to_volume(claims_df, "claims.csv")

In [0]:
display(dbutils.fs.ls(RAW_DATA_PATH))

In [0]:
members_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_DATA_PATH}/members.csv")

)

labs_test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_DATA_PATH}/labs.csv")
)

claims_test_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_DATA_PATH}/claims.csv")
)

print("Members count:", members_df.count())
print("Labs count:", labs_test_df.count())
print("Claims count:", claims_test_df.count())

display(members_df.limit(10))

In [0]:
dbutils.widgets.text("raw_data_path", RAW_DATA_PATH, "Raw Data Path")

raw_data_path = dbutils.widgets.get("raw_data_path")

print(f"Using raw data path: {raw_data_path}")

In [0]:
print("/Volumes/workspace/default/healthcare_project/healthcare-risk-data-lakehouse/raw/members.csv")